# Ask CIP: which open model could teach the next search model? (no OpenAI)

Ask sends its AI a request for 128 of the 161 golden questions (the rest it answers by names and rules). Here large open
models answer those same 128 requests on this GPU; back on the Mac each model's replies go through Ask exactly as the AI's
would, and get the same score as gpt-4.1 (155 of 161 right) and Ask with no AI (139 of 161). A model worth training the
next round from needs about 150 or more.

**Run:** Runtime → Change runtime type → **A100** (an L4 can only run the smaller 14B models) → Runtime → Run all → pick
`prompts.json` from `Downloads/cip-colab` when asked. About 30-40 minutes on an A100 (~6-8 units). At the end
`teacher-replies.zip` downloads: send it back. Nothing here uses an API key; a Hugging Face login is not needed (the
"HF_TOKEN" warning is harmless).

In [ ]:
# 1. The GPU, the libraries, the prompts
import importlib.metadata, importlib.util, json, os, re, shutil, subprocess, sys, time
import torch
assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → A100, then Run all again"
GPU = torch.cuda.get_device_properties(0)
MEMORY = GPU.total_memory / 1e9
print(f"{GPU.name}, {MEMORY:.0f} GB")
need = ["bitsandbytes"] if importlib.util.find_spec("bitsandbytes") is None else []
major, minor = (int(x) for x in importlib.metadata.version("transformers").split(".")[:2])
if (major, minor) < (4, 51):   # Qwen3 needs 4.51
    need.append("transformers>=4.51")
if need:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)
if not os.path.exists("prompts.json"):
    from google.colab import files
    print("Pick prompts.json (Downloads/cip-colab)")
    files.upload()
prompts = json.load(open("prompts.json"))
print(len(prompts), "requests, as Ask sends them")

In [ ]:
# 2. The models this GPU can hold (4-bit: a quarter of the memory, nearly the same answers). Each is tried from its
# ready 4-bit copy, else from Qwen's own files, made 4-bit while loading (slower to download)
if MEMORY > 70:
    MODELS = [("Qwen3-32B", ["unsloth/Qwen3-32B-bnb-4bit", "Qwen/Qwen3-32B"]),
              ("Qwen2.5-72B", ["unsloth/Qwen2.5-72B-Instruct-bnb-4bit", "Qwen/Qwen2.5-72B-Instruct"]),
              ("Qwen2.5-32B", ["unsloth/Qwen2.5-32B-Instruct-bnb-4bit", "Qwen/Qwen2.5-32B-Instruct"])]
elif MEMORY > 30:
    MODELS = [("Qwen3-32B", ["unsloth/Qwen3-32B-bnb-4bit", "Qwen/Qwen3-32B"]),
              ("Qwen2.5-32B", ["unsloth/Qwen2.5-32B-Instruct-bnb-4bit", "Qwen/Qwen2.5-32B-Instruct"])]
else:
    print("This GPU is too small for the 32B models: the 14B ones are tested. For the 32B, choose an A100.")
    MODELS = [("Qwen3-14B", ["unsloth/Qwen3-14B-bnb-4bit", "Qwen/Qwen3-14B"]),
              ("Qwen2.5-14B", ["unsloth/Qwen2.5-14B-Instruct-bnb-4bit", "Qwen/Qwen2.5-14B-Instruct"])]
print("will test:", ", ".join(name for name, _ in MODELS))

In [ ]:
# 3. Each model answers the 128 requests. OpenAI receives the reply format separately (a JSON schema); an open model
# gets the same keys in its instructions. Greedy (temperature 0) and at most 160 words of reply, as Ask asks of its AI
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

def load(repos):
    for repo in repos:
        try:
            tok = AutoTokenizer.from_pretrained(repo, padding_side="left")
            quant = None if "bnb-4bit" in repo else BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                                      bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
            model = AutoModelForCausalLM.from_pretrained(repo, device_map="cuda", torch_dtype=torch.bfloat16, quantization_config=quant)
            return repo, tok, model.eval()
        except Exception as exc:   # a copy that will not load: the next one
            print(f"  {repo} did not load ({type(exc).__name__}: {str(exc)[:120]}); trying the next")
            torch.cuda.empty_cache()
    return None, None, None

def chat(tok, p):
    system = p["system"] + "\nReply with one JSON object only, no other text, with exactly these keys: " + json.dumps(p["schema"]["properties"])
    return tok.apply_chat_template([{"role": "system", "content": system}, {"role": "user", "content": p["user"]}],
                                   tokenize=False, add_generation_prompt=True, enable_thinking=False)

def first_json(text):
    # the reply's JSON object (some models wrap it in ``` fences); the text itself when there is none, so Ask's own
    # check fails it as it would a broken AI reply
    found = re.search(r"\{.*\}", text, re.S)
    if found:
        try:
            json.loads(found.group(0))
            return found.group(0)
        except ValueError:
            pass
    return text

def keys_of(reply):
    try:
        found = json.loads(reply)
    except ValueError:
        return set()
    return set(found) if isinstance(found, dict) else set()

def answer(tok, model, texts):
    out, size, s = [], 16, 0
    while s < len(texts):
        try:
            enc = tok(texts[s:s + size], return_tensors="pt", padding=True).to("cuda")
            with torch.no_grad():
                made = model.generate(**enc, max_new_tokens=160, do_sample=False, temperature=None, top_p=None, top_k=None,
                                      pad_token_id=tok.pad_token_id or tok.eos_token_id)
            out += tok.batch_decode(made[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
            s += size
            print(f"  {min(s, len(texts))} of {len(texts)}", end=" | ")
        except torch.cuda.OutOfMemoryError:   # half as many at once, the same requests again
            enc = made = None
            torch.cuda.empty_cache()
            if size == 1:
                raise
            size //= 2
    print()
    return out

KEYS = set(prompts[0]["schema"]["properties"])
summary = []
for name, repos in MODELS:
    started = time.time()
    repo, tok, model = load(repos)
    if model is None:
        print(f"{name}: no copy loaded, skipped")
        continue
    print(f"{name} ({repo}) loaded in {time.time() - started:.0f}s")
    started = time.time()
    replies = [first_json(r) for r in answer(tok, model, [chat(tok, p) for p in prompts])]
    valid = sum(KEYS <= keys_of(r) for r in replies)
    json.dump({p["user"]: r for p, r in zip(prompts, replies)}, open(f"replies-{name}.json", "w"), indent=0)
    summary.append({"model": name, "repo": repo, "seconds": round(time.time() - started), "valid replies": valid})
    print(f"{name}: {valid} of {len(prompts)} replies in the right form · {time.time() - started:.0f}s")
    for p, r in list(zip(prompts, replies))[:3]:
        print("  ·", p["user"].splitlines()[0][:70], "→", r[:110])
    del model
    torch.cuda.empty_cache()
    shutil.rmtree(os.path.expanduser(f"~/.cache/huggingface/hub/models--{repo.replace('/', '--')}"), ignore_errors=True)   # disk for the next

In [ ]:
# 4. Everything in one file to send back
import zipfile
json.dump(summary, open("teacher-summary.json", "w"), indent=1)
with zipfile.ZipFile("teacher-replies.zip", "w") as z:
    for f in ["teacher-summary.json"] + [f"replies-{s['model']}.json" for s in summary]:
        z.write(f)
for s in summary:
    print(f"{s['model']}: {s['valid replies']} of {len(prompts)} in the right form, {s['seconds']}s")
from google.colab import files
files.download("teacher-replies.zip")
print("Done: send teacher-replies.zip. Runtime → Disconnect and delete runtime stops the units.")